In [1]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
from src.preprocessing.split_dataset import split_dataset

N_NEGATIVES = 100

In [2]:
df = pd.read_csv("../data/raw/interactions.csv")

# remove `ent_rate`, `time_type`, `group` columns
df = df.drop(columns=["ent_rate", "time_type", "group"])

df = df.rename(columns={"pfid": "user_id", "anchor_id": "streamer_id"})

print(f"Number of rows: {df.shape[0]}")
print(f"Number of columns: {df.shape[1]}")
print(f"Number of unique users: {df['user_id'].nunique()}")
print(f"Number of unique streamers: {df['streamer_id'].nunique()}")

Number of rows: 1599759
Number of columns: 8
Number of unique users: 25214
Number of unique streamers: 4053


# Split train, val, test interactions

split user's interacted streamer that should be in training, validation, and testing set

In [3]:
# train_df: all positive interactions
# val_df: 1 positive, N negatives
# test_df: 1 positive, N negatives

train_df, val_df, test_df = split_dataset(df, neg_per_pos=N_NEGATIVES)

# Feature engineering

build user & item aggregated features, then reapply it onto train, val, test set

In [4]:
# filter interactions for (user_id, streamer_id) that appears in train set
train_interaction_logs = df.merge(
    train_df[["user_id", "streamer_id"]],
    on=["user_id", "streamer_id"],
    how="inner"
)

In [5]:
# build user & item aggregated features, then reapply it onto train, val, test set
user_df = (
    train_interaction_logs.groupby("user_id").agg(
        u_watch_tot=("watch_ts", "sum"),
        u_watch_cnt=("watch_ts", "size"),
        u_gift_cnt =("consume_cnt", "sum"),
        u_gift_amt =("prod_total", "sum"),
        u_follow_cnt=("is_follow", "sum"),
    )
)

item_df = (
    train_interaction_logs.groupby("streamer_id").agg(
        i_watch_tot=("watch_ts", "sum"),
        i_watch_cnt=("watch_ts", "size"),
        i_unique_user=("user_id", "nunique"),
        i_live_cnt  =("live_cnt", "max"),      # already monthly total
        i_followers =("is_follow", "sum"),
        i_gift_amt  =("prod_total", "sum"),
    )
    .assign(
        i_watch_avg = lambda d: d.i_watch_tot / d.i_watch_cnt,
        i_pop_z     = lambda d: ((d.i_watch_cnt - d.i_watch_cnt.mean()) 
                                 / d.i_watch_cnt.std()),
    )
)

In [7]:
# sample N negatives for each user in train_df
# all_streamers = set(df["streamer_id"].unique())
streamers_in_train = set(train_df["streamer_id"].unique())
neg_samples = []

for user_id, group in train_df.groupby("user_id"):
    pos_streamers = set(group["streamer_id"])
    neg_candidates = list(streamers_in_train - pos_streamers)
    if len(neg_candidates) < N_NEGATIVES:
        sampled_negs = neg_candidates  # take all if not enough
    else:
        sampled_negs = np.random.choice(neg_candidates, N_NEGATIVES, replace=False)
    for neg in sampled_negs:
        neg_samples.append({"user_id": user_id, "streamer_id": neg, "label": 0})

neg_df = pd.DataFrame(neg_samples)
train_with_negs = pd.concat([train_df, neg_df], ignore_index=True)

In [8]:
# reapply user & item aggregated features onto train, val, test set
train_with_negs = train_with_negs.merge(user_df, on="user_id", how="left")
train_with_negs = train_with_negs.merge(item_df, on="streamer_id", how="left")

val_df = val_df.merge(user_df, on="user_id", how="left")
val_df = val_df.merge(item_df, on="streamer_id", how="left")

test_df = test_df.merge(user_df, on="user_id", how="left")
test_df = test_df.merge(item_df, on="streamer_id", how="left")

# fill NaN for cold-start streamers
val_df = val_df.fillna(val_df.mean())
test_df = test_df.fillna(test_df.mean())

In [9]:
train_with_negs.columns

Index(['user_id', 'streamer_id', 'label', 'u_watch_tot', 'u_watch_cnt',
       'u_gift_cnt', 'u_gift_amt', 'u_follow_cnt', 'i_watch_tot',
       'i_watch_cnt', 'i_unique_user', 'i_live_cnt', 'i_followers',
       'i_gift_amt', 'i_watch_avg', 'i_pop_z'],
      dtype='object')

In [10]:
FEATURE_COLS = [
    # user features
    'u_watch_tot', 'u_watch_cnt', 'u_gift_cnt', 'u_gift_amt', 'u_follow_cnt', 
    # item features
    'i_watch_cnt', 'i_unique_user', 'i_live_cnt', 'i_followers', 'i_gift_amt', 'i_watch_avg', 'i_pop_z'
]

In [11]:
bad_users = train_with_negs.groupby('user_id').label.nunique()
assert (bad_users == 2).all(), "some users miss pos or neg"

In [12]:
train_with_negs[train_with_negs["user_id"] == 1000235]

,user_id,streamer_id,label,u_watch_tot,u_watch_cnt,u_gift_cnt,u_gift_amt,u_follow_cnt,i_watch_tot,i_watch_cnt,i_unique_user,i_live_cnt,i_followers,i_gift_amt,i_watch_avg,i_pop_z
0,1000235,6017891,1,15.0,5,0.0,0.0,1.0,174081.0,881,462,56.0,13.0,19134.0,197.594779,0.666137
1,1000235,5734817,1,15.0,5,0.0,0.0,1.0,495781.0,1933,1061,44.0,32.0,16424.0,256.482669,2.067486
2,1000235,6078636,1,15.0,5,0.0,0.0,1.0,117804.0,1123,596,42.0,30.0,88781.0,104.901158,0.988500
817174,1000235,5663265,0,15.0,5,0.0,0.0,1.0,0.0,1,1,0.0,0.0,999.0,0.000000,-0.506094
817175,1000235,6216813,0,15.0,5,0.0,0.0,1.0,52748.0,314,177,35.0,9.0,608.0,167.987261,-0.089153
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
817269,1000235,6716584,0,15.0,5,0.0,0.0,1.0,241289.0,1190,633,43.0,43.0,30449.0,202.763866,1.077750
817270,1000235,3261803,0,15.0,5,0.0,0.0,1.0,47686.0,206,112,11.0,8.0,715.0,231.485437,-0.233018
817271,1000235,3349459,0,15.0,5,0.0,0.0,1.0,23699.0,74,37,3.0,0.0,0.0,320.256757,-0.408853
817272,1000235,3629052,0,15.0,5,0.0,0.0,1.0,0.0,1,1,0.0,0.0,100.0,0.000000,-0.506094


# Train LightGBMRanker

In [13]:
from lightgbm import LGBMRanker, early_stopping
from sklearn.preprocessing import StandardScaler

In [14]:
train_with_negs.head()

,user_id,streamer_id,label,u_watch_tot,u_watch_cnt,u_gift_cnt,u_gift_amt,u_follow_cnt,i_watch_tot,i_watch_cnt,i_unique_user,i_live_cnt,i_followers,i_gift_amt,i_watch_avg,i_pop_z
0,1000235,6017891,1,15.0,5,0.0,0.0,1.0,174081.0,881,462,56.0,13.0,19134.0,197.594779,0.666137
1,1000235,5734817,1,15.0,5,0.0,0.0,1.0,495781.0,1933,1061,44.0,32.0,16424.0,256.482669,2.067486
2,1000235,6078636,1,15.0,5,0.0,0.0,1.0,117804.0,1123,596,42.0,30.0,88781.0,104.901158,0.988500
3,1001050,1248228,1,33.0,3,0.0,0.0,0.0,489909.0,3023,1549,21.0,82.0,107344.0,162.060536,3.519454
4,1001050,6510054,1,33.0,3,0.0,0.0,0.0,332555.0,1612,891,51.0,33.0,34927.0,206.299628,1.639888


In [15]:
# Sort rows so group order = row order
train_with_negs.sort_values('user_id', inplace=True)
val_df.sort_values('user_id', inplace=True)

# scale numeric columns (tree prefers but not mandatory)
scaler = StandardScaler().fit(train_with_negs[FEATURE_COLS])

X_train = scaler.transform(train_with_negs[FEATURE_COLS])
y_train = train_with_negs["label"].values

X_val = scaler.transform(val_df[FEATURE_COLS])
y_val = val_df["label"].values

X_test = scaler.transform(test_df[FEATURE_COLS])
y_test = test_df["label"].values

train_group = train_with_negs.groupby("user_id").size().to_numpy()
val_group = val_df.groupby("user_id").size().to_numpy()

In [16]:
ranker = LGBMRanker(
    objective="lambdarank",
    metric="ndcg",
    ndcg_eval_at=[10,20,50],
    num_leaves=63,
    n_estimators=100,
    learning_rate=0.05,
)

callbacks = [early_stopping(stopping_rounds=30, verbose=True)]

ranker.fit(
    X_train, y_train,
    group=train_group,
    eval_set=[(X_val, y_val)],
    eval_group=[val_group], 
    callbacks=callbacks,
)

/opt/homebrew/Caskroom/miniconda/base/envs/ml_env/lib/python3.9/site-packages/lightgbm/sklearn.py:861: UserWarning: Found 'ndcg_eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.028029 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2606
[LightGBM] [Info] Number of data points in the train set: 2964874, number of used features: 12
Training until validation scores don't improve for 30 rounds
Did not meet early stopping. Best iteration is:
[100]	valid_0's ndcg@10: 0.439607	valid_0's ndcg@20: 0.485358	valid_0's ndcg@50: 0.506457


LGBMRanker(learning_rate=0.05, metric='ndcg', ndcg_eval_at=[10, 20, 50],
           num_leaves=63, objective='lambdarank')

# Evaluation

In [17]:
scores = ranker.predict(X_test)
test_df["score"] = scores

# split rows by user
sizes  = test_df.groupby("user_id").size().to_numpy() # group vector
idx    = np.cumsum(sizes)[:-1] # cumulative sume, indicating where each group ends
groups = np.split(test_df.to_numpy(), idx) # list of arrays

# col indices for speed
LABEL_COL  = test_df.columns.get_loc("label")
SCORE_COL  = test_df.columns.get_loc("score")

ranks = [] # # rank (1-based) of the positive per user
for group in groups:
    # sort descending by score inside this user group
    group_sorted = group[np.argsort(-group[:, SCORE_COL])]
    # index of the unique positive (label=1) in the sorted group
    pos_rank = np.where(group_sorted[:, LABEL_COL] == 1)[0][0] + 1
    ranks.append(pos_rank)  # store the rank (1-based)
ranks = np.asarray(ranks) # shape [n_users]

/opt/homebrew/Caskroom/miniconda/base/envs/ml_env/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRanker was fitted with feature names
  warnings.warn(
/opt/homebrew/Caskroom/miniconda/base/envs/ml_env/lib/python3.9/site-packages/lightgbm/sklearn.py:861: UserWarning: Found 'ndcg_eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


In [18]:
# precision is not needed since the denominator is the same with recall in leave-one-out setting

def recall_at_k(r, k):
    return (r <= k).mean()

def ndcg_at_k(r, k):
    return np.where(r <= k, 1 / np.log2(r + 1), 0).mean()

def mrr_at_k(r, k):
    return np.where(r <= k, 1 / r, 0).mean()

KS = (10, 20, 50)
for k in KS:
    print(f"k={k:<2}  "
          f"Recall(Hit rate) {recall_at_k(ranks,k):.4f}  "
          f"nDCG {ndcg_at_k(ranks,k):.4f}  "
          f"MRR  {mrr_at_k(ranks,k):.4f}")

k=10  Recall(Hit rate) 0.7137  nDCG 0.4396  MRR  0.3553
k=20  Recall(Hit rate) 0.8947  nDCG 0.4856  MRR  0.3680
k=50  Recall(Hit rate) 0.9983  nDCG 0.5070  MRR  0.3718


# Analysis

Model is performing too well: possible issues:
* negatives are too easy (streamer that user has never interacted with)

In [19]:
print("rows:", len(ranks))
print("min rank of positive:", ranks.min(), "max rank:", ranks.max())

rows: 21477
min rank of positive: 1 max rank: 101


## Inspect top features

Seems to recommend the most popular streamer

In [20]:
imp = ranker.booster_.feature_importance(importance_type="gain")
for feat, gain in sorted(zip(FEATURE_COLS, imp), key=lambda x: -x[1])[:10]:
    print(feat, gain)

i_watch_cnt 1114211.8174271584
i_unique_user 217310.69994783401
u_watch_cnt 47301.14569711685
i_watch_avg 23940.35956287384
u_watch_tot 20767.332223415375
i_followers 14570.24348115921
i_gift_amt 14412.880065441132
i_live_cnt 10917.025649309158
u_follow_cnt 6403.644111156464
u_gift_amt 4667.067013978958


## Hold-out cold-item split

a special test set where all positives are streamers that never appear in training. 

In [21]:
# ids of every streamer that appears in train interactions
train_items = set(train_with_negs["streamer_id"].unique())

# positives in the existing test split
test_pos = test_df.loc[test_df["label"] == 1, ["user_id", "streamer_id"]]

# cold positives = streamer_id not seen in train
cold_pos = test_pos[~test_pos["streamer_id"].isin(train_items)]
cold_item_set = set(cold_pos["streamer_id"])
print("cold positives:", len(cold_pos), "unique cold items:",
      len(cold_item_set))


cold positives: 27 unique cold items: 27


In [22]:
# keep only the user groups that have a cold positive
cold_users = set(cold_pos["user_id"])

# filter the whole test_df to those users only
cold_test = test_df[test_df["user_id"].isin(cold_users)].copy()

In [23]:
cold_test

,user_id,streamer_id,label,u_watch_tot,u_watch_cnt,u_gift_cnt,u_gift_amt,u_follow_cnt,i_watch_tot,i_watch_cnt,i_unique_user,i_live_cnt,i_followers,i_gift_amt,i_watch_avg,i_pop_z,score
21210,1042905,1008732,1,1894.0,20,10.0,1699.0,0.0,75402.802763,383.8677,206.605754,12.440223,7.606808,19355.24991,109.855398,0.003916,-1.859626
21211,1042905,2616828,0,1894.0,20,10.0,1699.0,0.0,256924.000000,1413.0000,773.000000,21.000000,29.000000,41364.00000,181.828733,1.374804,-0.027821
21212,1042905,6732823,0,1894.0,20,10.0,1699.0,0.0,0.000000,1.0000,1.000000,0.000000,0.000000,0.00000,0.000000,-0.506094,-5.015576
21213,1042905,6700180,0,1894.0,20,10.0,1699.0,0.0,0.000000,1.0000,1.000000,0.000000,0.000000,0.00000,0.000000,-0.506094,-5.015576
21214,1042905,1995418,0,1894.0,20,10.0,1699.0,0.0,380734.000000,1869.0000,1035.000000,29.000000,18.000000,28234.00000,203.710005,1.982232,0.197548
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1790624,6813671,4194632,0,4.0,2,0.0,0.0,0.0,163.000000,6.0000,3.000000,1.000000,1.000000,0.00000,27.166667,-0.499434,-5.179760
1790625,6813671,6687479,0,4.0,2,0.0,0.0,0.0,110517.000000,447.0000,238.000000,23.000000,14.000000,22856.00000,247.241611,0.088014,-1.580146
1790626,6813671,1581660,0,4.0,2,0.0,0.0,0.0,122778.000000,459.0000,250.000000,14.000000,13.000000,13255.00000,267.490196,0.103999,-1.577813
1790627,6813671,6787138,0,4.0,2,0.0,0.0,0.0,75402.802763,383.8677,206.605754,12.440223,7.606808,19355.24991,109.855398,0.003916,-1.876509


In [24]:
X_cold_test = scaler.transform(cold_test[FEATURE_COLS])
y_cold_test = cold_test["label"].values

scores = ranker.predict(X_cold_test)
cold_test["score"] = scores

sizes = cold_test.groupby("user_id").size().to_numpy()  # group vector
idx = np.cumsum(sizes)[:-1]  # cumulative sum, indicating where each group ends
groups = np.split(cold_test.to_numpy(), idx)  # list of arrays

ranks = []  # rank (1-based) of the positive per user
for group in groups:
    # sort descending by score inside this user group
    group_sorted = group[np.argsort(-group[:, SCORE_COL])]
    # index of the unique positive (label=1) in the sorted group
    pos_rank = np.where(group_sorted[:, LABEL_COL] == 1)[0][0] + 1
    ranks.append(pos_rank)  # store the rank (1-based)
ranks = np.asarray(ranks)  # shape [n_users]

/opt/homebrew/Caskroom/miniconda/base/envs/ml_env/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRanker was fitted with feature names
  warnings.warn(
/opt/homebrew/Caskroom/miniconda/base/envs/ml_env/lib/python3.9/site-packages/lightgbm/sklearn.py:861: UserWarning: Found 'ndcg_eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


In [25]:
KS = (10, 20, 50)
for k in KS:
    print(f"k={k:<2}  "
          f"Recall(Hit rate) {recall_at_k(ranks,k):.4f}  "
          f"nDCG {ndcg_at_k(ranks,k):.4f}  "
          f"MRR  {mrr_at_k(ranks,k):.4f}")

k=10  Recall(Hit rate) 0.0000  nDCG 0.0000  MRR  0.0000
k=20  Recall(Hit rate) 0.0000  nDCG 0.0000  MRR  0.0000
k=50  Recall(Hit rate) 0.9259  nDCG 0.1854  MRR  0.0301
